# Metaculus Medaled User Predictions Aggregator

This notebook fetches comments with predictions from a Metaculus question, filters for comments from users who have at least one medal, and provides aggregate prediction statistics.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dishuk13/gdp-dashboard/blob/main/scripts/metaculus_medaled_predictions.ipynb)

## Features

- 📊 Fetches all comments for a given Metaculus question
- 🏅 Identifies users with medals (gold, silver, or bronze)
- 🎯 Extracts predictions from comments made by medaled users
- 📈 Calculates aggregate statistics and visualizations
- 💾 Exports results to CSV

## How to Use

1. Run the setup cells to install dependencies and import libraries
2. Configure your question ID or URL in the configuration cell
3. (Optional) Add your Metaculus API token for authenticated requests
4. Run all cells to fetch and analyze the data

## Setup

Install required packages (only needed in Colab)

In [ ]:
# Uncomment the following line if running in Google Colab
# !pip install requests pandas matplotlib seaborn

### Import Libraries

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statistics
import re
from typing import List, Dict, Any, Optional
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully!")

## Configuration

Set your Metaculus question ID/URL and optional API token here.

In [ ]:
# ============ CONFIGURATION ============

# Enter your Metaculus question ID or full URL
# Examples:
#   - "578"
#   - "https://www.metaculus.com/questions/578/"
QUESTION_INPUT = "578"

# Optional: Add your Metaculus API token for authenticated requests
# Leave as None if you don't have one
API_TOKEN = None

# ========================================

## Metaculus API Wrapper

Define the API client class for interacting with Metaculus.

In [ ]:
class MetaculusAPI:
    """Wrapper for Metaculus API interactions."""

    BASE_URL = "https://www.metaculus.com/api2"

    def __init__(self, api_token: Optional[str] = None):
        """
        Initialize the API client.

        Args:
            api_token: Optional API token for authenticated requests
        """
        self.session = requests.Session()
        if api_token:
            self.session.headers.update({
                'Authorization': f'Token {api_token}'
            })

    def get_question(self, question_id: int) -> Dict[str, Any]:
        """
        Fetch question details from Metaculus API.

        Args:
            question_id: The question ID

        Returns:
            Question data dictionary
        """
        url = f"{self.BASE_URL}/questions/{question_id}/"
        response = self.session.get(url)
        response.raise_for_status()
        return response.json()

    def get_comments(self, question_id: int, limit: int = 100) -> List[Dict[str, Any]]:
        """
        Fetch all comments for a question.
        Tries multiple endpoint approaches to handle API variations.

        Args:
            question_id: The question ID
            limit: Number of comments per request

        Returns:
            List of comment dictionaries
        """
        # Try multiple approaches to fetch comments
        approaches = [
            # Approach 1: on_post parameter (most likely to work)
            ('on_post', f"{self.BASE_URL}/comments/", {'on_post': question_id, 'limit': limit}),
            # Approach 2: question parameter
            ('question', f"{self.BASE_URL}/comments/", {'question': question_id, 'limit': limit}),
            # Approach 3: nested endpoint under posts
            ('nested_posts', f"{self.BASE_URL}/posts/{question_id}/comments/", {'limit': limit}),
        ]

        last_error = None

        for approach_name, url, params in approaches:
            try:
                all_comments = []
                current_params = params.copy()

                while True:
                    response = self.session.get(url, params=current_params)
                    response.raise_for_status()
                    data = response.json()

                    # Handle both list and dict responses
                    if isinstance(data, list):
                        all_comments.extend(data)
                        break  # Lists don't have pagination
                    elif isinstance(data, dict):
                        results = data.get('results', [])
                        all_comments.extend(results)

                        # Check if there's a next page
                        next_url = data.get('next')
                        if not next_url:
                            break

                        # Extract cursor from next URL
                        if 'cursor=' in next_url:
                            cursor = next_url.split('cursor=')[1].split('&')[0]
                            current_params['cursor'] = cursor
                            # Remove original filter param after first request
                            current_params.pop('question', None)
                            current_params.pop('on_post', None)
                        else:
                            break
                    else:
                        break

                # If we got here without an exception, it worked!
                if all_comments or approach_name == approaches[-1][0]:
                    # Success! or last attempt
                    return all_comments

            except requests.exceptions.HTTPError as e:
                last_error = e
                if e.response.status_code == 405:
                    # Method not allowed, try next approach
                    continue
                elif e.response.status_code == 404:
                    # Not found, try next approach
                    continue
                else:
                    # Other HTTP error, might be more serious
                    raise
            except Exception as e:
                last_error = e
                continue

        # If all approaches failed, raise the last error
        if last_error:
            print(f"\n⚠️  Warning: Could not fetch comments using standard approaches.")
            print(f"   Last error: {last_error}")
            print(f"   This may mean comments are not accessible via the API,")
            print(f"   or the API structure has changed.")
            print(f"   Run 'python scripts/test_metaculus_api.py {question_id}' for diagnostics.")

        return []

    def get_user(self, user_id: int) -> Dict[str, Any]:
        """
        Fetch user details from Metaculus API.

        Args:
            user_id: The user ID

        Returns:
            User data dictionary
        """
        url = f"{self.BASE_URL}/users/{user_id}/"
        response = self.session.get(url)
        response.raise_for_status()
        return response.json()

print("✅ MetaculusAPI class defined!")

## Helper Functions

In [ ]:
def extract_question_id(input_str: str) -> int:
    """
    Extract question ID from URL or ID string.

    Args:
        input_str: Question ID or Metaculus question URL

    Returns:
        Question ID as integer

    Raises:
        ValueError: If question ID cannot be extracted
    """
    # If it's already a number
    if str(input_str).isdigit():
        return int(input_str)

    # Try to extract from URL
    match = re.search(r'/questions/(\d+)/', str(input_str))
    if match:
        return int(match.group(1))

    raise ValueError(f"Could not extract question ID from: {input_str}")


def has_medals(user_data: Dict[str, Any]) -> bool:
    """
    Check if a user has at least one medal.

    Args:
        user_data: User data dictionary from API

    Returns:
        True if user has at least one medal, False otherwise
    """
    # Check if there's a medals field
    if 'medals' in user_data:
        medals = user_data['medals']
        if isinstance(medals, list) and len(medals) > 0:
            return True
        if isinstance(medals, dict) and any(medals.values()):
            return True

    # Check medal counts
    medal_fields = ['gold_medals', 'silver_medals', 'bronze_medals', 'total_medals']
    for field in medal_fields:
        if field in user_data and user_data[field] and user_data[field] > 0:
            return True

    # Check if there's a medal_count field
    if 'medal_count' in user_data and user_data['medal_count'] > 0:
        return True

    return False


def extract_prediction_from_comment(comment: Dict[str, Any]) -> Optional[float]:
    """
    Extract prediction value from a comment if it contains one.

    Args:
        comment: Comment data dictionary

    Returns:
        Prediction value (0-1 for binary questions) or None if no prediction found
    """
    # Check if comment has a prediction_snapshot field
    if 'prediction_snapshot' in comment and comment['prediction_snapshot']:
        snapshot = comment['prediction_snapshot']

        # For binary questions, look for probability
        if isinstance(snapshot, dict):
            if 'probability' in snapshot:
                return snapshot['probability']
            elif 'prediction' in snapshot:
                return snapshot['prediction']
            elif 'q2' in snapshot:  # q2 is median
                return snapshot['q2']
        elif isinstance(snapshot, (int, float)):
            return float(snapshot)

    # Check for prediction in comment metadata
    if 'prediction' in comment and comment['prediction'] is not None:
        pred = comment['prediction']
        if isinstance(pred, dict) and 'probability' in pred:
            return pred['probability']
        elif isinstance(pred, (int, float)):
            return float(pred)

    return None


def aggregate_predictions(predictions: List[float]) -> Dict[str, float]:
    """
    Calculate aggregate statistics from predictions.

    Args:
        predictions: List of prediction values

    Returns:
        Dictionary with aggregate statistics
    """
    if not predictions:
        return {}

    return {
        'mean': statistics.mean(predictions),
        'median': statistics.median(predictions),
        'min': min(predictions),
        'max': max(predictions),
        'count': len(predictions),
        'stdev': statistics.stdev(predictions) if len(predictions) > 1 else 0.0
    }

print("✅ Helper functions defined!")

## Fetch and Process Data

Now let's fetch the question, comments, and filter by medaled users.

In [ ]:
# Extract question ID
question_id = extract_question_id(QUESTION_INPUT)
print(f"📋 Processing question ID: {question_id}")

# Initialize API client
api = MetaculusAPI(API_TOKEN)

# Fetch question details
print("\n🔍 Fetching question details...")
question = api.get_question(question_id)
print(f"   Title: {question.get('title', 'Unknown')}")
print(f"   Type: {question.get('type', 'Unknown')}")
print(f"   URL: https://www.metaculus.com/questions/{question_id}/")

# Fetch comments
print("\n💬 Fetching comments...")
comments = api.get_comments(question_id)
print(f"   Found {len(comments)} total comments")

In [ ]:
# Filter comments by medaled users and extract predictions
print("\n🏅 Filtering comments by medaled users...")

medaled_user_predictions = []
medaled_users = {}
user_cache = {}  # Cache user data to avoid repeated API calls
prediction_details = []  # Store details for DataFrame

for i, comment in enumerate(comments):
    if i % 10 == 0:
        print(f"   Processing comment {i+1}/{len(comments)}...", end='\r')
    
    author_id = comment.get('author')
    if not author_id:
        continue

    # Check cache first
    if author_id not in user_cache:
        try:
            user_data = api.get_user(author_id)
            user_cache[author_id] = user_data
        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 404:
                # User not found, skip
                user_cache[author_id] = None
                continue
            raise

    user_data = user_cache[author_id]
    if user_data is None:
        continue

    # Check if user has medals
    if has_medals(user_data):
        username = user_data.get('username', f'User {author_id}')
        medaled_users[author_id] = username

        # Try to extract prediction from comment
        prediction = extract_prediction_from_comment(comment)
        if prediction is not None:
            medaled_user_predictions.append(prediction)
            
            # Store details
            prediction_details.append({
                'username': username,
                'user_id': author_id,
                'prediction': prediction,
                'comment_id': comment.get('id'),
                'created_time': comment.get('created_time', '')
            })

print(f"\n   Found {len(medaled_users)} unique users with medals")
print(f"   Found {len(medaled_user_predictions)} predictions from medaled users")

## Create Results DataFrame

In [ ]:
# Create DataFrame with prediction details
if prediction_details:
    df = pd.DataFrame(prediction_details)
    df = df.sort_values('prediction', ascending=False).reset_index(drop=True)
    
    print("\n📊 Prediction Details:")
    print(df.to_string(index=False))
else:
    df = pd.DataFrame()
    print("\n⚠️  No predictions found in comments")

## Aggregate Statistics

In [ ]:
if medaled_user_predictions:
    aggregates = aggregate_predictions(medaled_user_predictions)
    
    print("\n" + "="*60)
    print("📈 AGGREGATE PREDICTIONS FROM MEDALED USERS")
    print("="*60)
    print(f"Count:          {aggregates['count']}")
    print(f"Mean:           {aggregates['mean']:.4f}")
    print(f"Median:         {aggregates['median']:.4f}")
    print(f"Min:            {aggregates['min']:.4f}")
    print(f"Max:            {aggregates['max']:.4f}")
    print(f"Std Deviation:  {aggregates['stdev']:.4f}")
    print("="*60)
    
    # Create summary DataFrame
    summary_df = pd.DataFrame([aggregates])
    display(summary_df)
else:
    print("\n⚠️  No predictions found to aggregate")
    aggregates = None

## Visualizations

In [ ]:
if medaled_user_predictions and len(medaled_user_predictions) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'Predictions from Medaled Users\n{question.get("title", "")}', 
                 fontsize=14, fontweight='bold')
    
    # 1. Histogram
    axes[0, 0].hist(medaled_user_predictions, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0, 0].axvline(aggregates['mean'], color='red', linestyle='--', linewidth=2, label=f"Mean: {aggregates['mean']:.4f}")
    axes[0, 0].axvline(aggregates['median'], color='green', linestyle='--', linewidth=2, label=f"Median: {aggregates['median']:.4f}")
    axes[0, 0].set_xlabel('Prediction Value')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Distribution of Predictions')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Box plot
    axes[0, 1].boxplot(medaled_user_predictions, vert=True)
    axes[0, 1].set_ylabel('Prediction Value')
    axes[0, 1].set_title('Box Plot')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].set_xticklabels(['Predictions'])
    
    # 3. Scatter plot with index
    axes[1, 0].scatter(range(len(medaled_user_predictions)), medaled_user_predictions, alpha=0.6, s=100, color='coral')
    axes[1, 0].axhline(aggregates['mean'], color='red', linestyle='--', linewidth=2, label=f"Mean: {aggregates['mean']:.4f}")
    axes[1, 0].axhline(aggregates['median'], color='green', linestyle='--', linewidth=2, label=f"Median: {aggregates['median']:.4f}")
    axes[1, 0].set_xlabel('Prediction Index')
    axes[1, 0].set_ylabel('Prediction Value')
    axes[1, 0].set_title('Individual Predictions')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Summary statistics text
    axes[1, 1].axis('off')
    summary_text = f"""
    Summary Statistics
    ==================
    
    Count:     {aggregates['count']}
    Mean:      {aggregates['mean']:.4f}
    Median:    {aggregates['median']:.4f}
    Min:       {aggregates['min']:.4f}
    Max:       {aggregates['max']:.4f}
    Std Dev:   {aggregates['stdev']:.4f}
    Range:     {aggregates['max'] - aggregates['min']:.4f}
    
    Question ID: {question_id}
    Total Comments: {len(comments)}
    Medaled Users: {len(medaled_users)}
    """
    axes[1, 1].text(0.1, 0.5, summary_text, fontsize=12, family='monospace',
                    verticalalignment='center')
    
    plt.tight_layout()
    plt.show()
    
    # Save the figure
    fig.savefig(f'metaculus_predictions_{question_id}.png', dpi=300, bbox_inches='tight')
    print(f"\n💾 Figure saved as 'metaculus_predictions_{question_id}.png'")
else:
    print("\n⚠️  Not enough data to create visualizations")

## Export Results

Export the results to CSV for further analysis.

In [ ]:
if not df.empty:
    # Export predictions
    filename = f'metaculus_predictions_{question_id}.csv'
    df.to_csv(filename, index=False)
    print(f"✅ Predictions exported to '{filename}'")
    
    # Export summary statistics
    if aggregates:
        summary_filename = f'metaculus_summary_{question_id}.csv'
        summary_df = pd.DataFrame([aggregates])
        summary_df.to_csv(summary_filename, index=False)
        print(f"✅ Summary statistics exported to '{summary_filename}'")
    
    # Show download links in Colab
    try:
        from google.colab import files
        print("\n📥 Download files:")
        files.download(filename)
        if aggregates:
            files.download(summary_filename)
        files.download(f'metaculus_predictions_{question_id}.png')
    except ImportError:
        print("\n💡 Not running in Colab. Files saved to current directory.")
else:
    print("\n⚠️  No data to export")

## Notes and Limitations

### Important Notes:

1. **API Structure**: The Metaculus API is still evolving, and some features may change.

2. **Prediction Snapshots**: Not all comments contain prediction snapshots. This script extracts predictions where available.

3. **Medal Detection**: The script checks multiple fields in user data to identify medals. The exact structure may vary.

4. **Rate Limiting**: Making many API calls may trigger rate limiting. Using an API token can help.

5. **Question Types**: This notebook works best with binary questions. Other question types (numeric, date, multiple choice) may require adjustments.

### Troubleshooting:

- **No predictions found**: Some questions store predictions separately from comments
- **API errors**: Check your API token and network connection
- **Empty results**: The question may not have medaled users commenting with predictions

### Resources:

- [Metaculus Website](https://www.metaculus.com/)
- [Metaculus API Documentation](https://www.metaculus.com/api/)
- [Metaculus FAQ](https://www.metaculus.com/faq/)
- [Medals FAQ](https://www.metaculus.com/help/medals-faq/)